In [1]:
# import io
# import zipfile
# import requests
# import frontmatter

# url = 'https://github.com/DataTalksClub/data-engineering-zoomcamp/archive/refs/heads/main.zip'
# resp = requests.get(url)


# repository_data = []
# zf = zipfile.ZipFile(io.BytesIO(resp.content))

# for file_info in zf.infolist():
#     filename = file_info.filename.lower()
#     if not (filename.endswith('.md') or filename.endswith('.mdx')):
#         continue
#     with zf.open(file_info) as f_in:
#         content = f_in.read()
#         post = frontmatter.loads(content)
#         data = post.to_dict()
#         data['filename'] = filename
#         repository_data.append(data)

# zf.close()

# print(repository_data[1])


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: DataTalksClub
        repo_name: data-engineering-zoomcamp
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/archive/refs/heads/main.zip'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [3]:
dtc_data_engineering_zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"DEZ documents: {len(dtc_data_engineering_zoomcamp)}")
print(f"Evidently documents: {len(evidently_docs)}")


DEZ documents: 143
Evidently documents: 95


In [4]:
from openai import OpenAI

openai_client = OpenAI()


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text


In [5]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [6]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections


In [7]:
from tqdm.auto import tqdm

evidently_chunks = []

for doc in tqdm(evidently_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

  0%|          | 0/95 [00:00<?, ?it/s]

In [8]:
# def sliding_window(seq, size, step):
#     if size <= 0 or step <= 0:
#         raise ValueError("size and step must be positive")

#     n = len(seq)
#     result = []
#     for i in range(0, n, step):
#         chunk = seq[i:i+size]
#         result.append({'start': i, 'chunk': chunk})
#         if i + size >= n:
#             break

#     return result


In [9]:
# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     chunks = sliding_window(doc_content, 2000, 1000)
#     for chunk in chunks:
#         chunk.update(doc_copy)
#     evidently_chunks.extend(chunks)

In [10]:
# import re
# text = evidently_docs[45]['content']
# paragraphs = re.split(r"\n\s*\n", text.strip())


In [11]:
# import re

# def split_markdown_by_level(text, level=2):
#     """
#     Split markdown text by a specific header level.
    
#     :param text: Markdown text as a string
#     :param level: Header level to split on
#     :return: List of sections as strings
#     """
#     # This regex matches markdown headers
#     # For level 2, it matches lines starting with "## "
#     header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
#     pattern = re.compile(header_pattern, re.MULTILINE)

#     # Split and keep the headers
#     parts = pattern.split(text)
    
#     sections = []
#     for i in range(1, len(parts), 3):
#         # We step by 3 because regex.split() with
#         # capturing groups returns:
#         # [before_match, group1, group2, after_match, ...]
#         # here group1 is "## ", group2 is the header text
#         header = parts[i] + parts[i+1]  # "## " + "Title"
#         header = header.strip()

#         # Get the content after this header
#         content = ""
#         if i+2 < len(parts):
#             content = parts[i+2].strip()

#         if content:
#             section = f'{header}\n\n{content}'
#         else:
#             section = header
#         sections.append(section)
    
#     return sections


In [12]:
# sections = split_markdown_by_level(text, level=2)

In [13]:
# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = split_markdown_by_level(doc_content, level=2)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks.append(section_doc)


In [14]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)


In [15]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)


In [16]:
dtc_data_engineering_zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')

de_dtc_data_engineering_zoomcamp = [d for d in dtc_data_engineering_zoomcamp if 'data-engineering' in d['filename']]

data_engineering_zoomcamp_index = Index(
    text_fields=["content", "filename"],
    keyword_fields=[]
)

data_engineering_zoomcamp_index.fit(de_dtc_data_engineering_zoomcamp)


In [17]:
print(de_dtc_data_engineering_zoomcamp[2])

{'content': '# Virtual Environments and Data Pipelines\n\n**[↑ Up](README.md)** | **[← Previous](01-introduction.md)** | **[Next →](03-dockerizing-pipeline.md)**\n\nA **data pipeline** is a service that receives data as input and outputs more data. For example, reading a CSV file, transforming the data somehow and storing it as a table in a PostgreSQL database.\n\n```mermaid\ngraph LR\n    A[CSV File] --> B[Data Pipeline]\n    B --> C[Parquet File]\n    B --> D[PostgreSQL Database]\n    B --> E[Data Warehouse]\n    style B fill:#4CAF50,stroke:#333,stroke-width:2px,color:#fff\n```\n\nIn this workshop, we\'ll build pipelines that:\n- Download CSV data from the web\n- Transform and clean the data with pandas\n- Load it into PostgreSQL for querying\n- Process data in chunks to handle large files\n\n## Creating a Simple Pipeline\n\nLet\'s create an example pipeline. First, create a directory `pipeline` and inside, create a file  `pipeline.py`:\n\n```python\nimport sys\nprint("arguments", sy

In [18]:
import json
pretty_json = json.dumps(de_dtc_data_engineering_zoomcamp[2], indent=4)
print(pretty_json)

{
    "content": "# Virtual Environments and Data Pipelines\n\n**[\u2191 Up](README.md)** | **[\u2190 Previous](01-introduction.md)** | **[Next \u2192](03-dockerizing-pipeline.md)**\n\nA **data pipeline** is a service that receives data as input and outputs more data. For example, reading a CSV file, transforming the data somehow and storing it as a table in a PostgreSQL database.\n\n```mermaid\ngraph LR\n    A[CSV File] --> B[Data Pipeline]\n    B --> C[Parquet File]\n    B --> D[PostgreSQL Database]\n    B --> E[Data Warehouse]\n    style B fill:#4CAF50,stroke:#333,stroke-width:2px,color:#fff\n```\n\nIn this workshop, we'll build pipelines that:\n- Download CSV data from the web\n- Transform and clean the data with pandas\n- Load it into PostgreSQL for querying\n- Process data in chunks to handle large files\n\n## Creating a Simple Pipeline\n\nLet's create an example pipeline. First, create a directory `pipeline` and inside, create a file  `pipeline.py`:\n\n```python\nimport sys\npri

In [19]:
print(results)

[{'title': 'RAG evaluation dataset', 'description': 'Synthetic data for RAG.', 'filename': 'docs-main/synthetic-data/rag_data.mdx', 'section': '## Creating a RAG Test Dataset\n\nYou can generate a ground truth RAG dataset from your data source using the following steps.'}, {'title': 'RAG evaluation dataset', 'description': 'Synthetic data for RAG.', 'filename': 'docs-main/synthetic-data/rag_data.mdx', 'section': '### 4. Save the Dataset\n\nOnce you are finished, store the dataset. You can download it as a CSV file or access it via the Python API using the dataset ID for evaluation.\n\n<Info>\n**Dataset API.** How to work with [Evidently datasets](/docs/platform/datasets_overview).\n</Info>'}, {'title': 'RAG evaluation dataset', 'description': 'Synthetic data for RAG.', 'filename': 'docs-main/synthetic-data/rag_data.mdx', 'section': '### 3. Review the Test Cases\n\nYou can preview and refine the generated dataset.\n\n![Review Test Cases](images/synthetic/synthetic_data_rag_example_resul

In [20]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [21]:
record = de_dtc_data_engineering_zoomcamp[2]
text = record['content'] + ' ' + record['filename']
v_doc = embedding_model.encode(text)


In [22]:
query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)


In [23]:
similarity = v_query.dot(v_doc)

In [24]:
from tqdm.auto import tqdm
import numpy as np

dez_embeddings = []

for d in tqdm(de_dtc_data_engineering_zoomcamp):
    text = d['content'] + ' ' + d['filename']
    v = embedding_model.encode(text)
    dez_embeddings.append(v)

dez_embeddings = np.array(dez_embeddings)


  0%|          | 0/143 [00:00<?, ?it/s]

In [25]:
print(v)

[ 6.47748262e-02  1.27556259e-02 -2.84837792e-03 -9.62572452e-03
  1.75142512e-02  2.91470569e-02 -4.90807481e-02  3.07970829e-02
 -1.50757506e-02 -9.63438451e-02 -7.38240487e-04 -1.47899864e-02
  3.39498743e-02 -1.76068190e-02  4.84311543e-02  1.12512326e-02
  1.29389800e-02 -2.46094782e-02  4.54633050e-02  2.53485087e-02
  2.41185538e-02  3.43885971e-03 -2.86714900e-02  4.27458473e-02
 -2.88643818e-02  2.54616067e-02  1.01638371e-02  3.10219359e-02
 -4.70793713e-03 -6.73087612e-02 -2.08235998e-02 -1.74780674e-02
 -6.55238982e-03 -3.24590951e-02  5.06484918e-02 -2.27743899e-03
 -5.76596297e-02  2.03555003e-02 -2.32229009e-02 -3.70635055e-02
 -3.28592025e-02  3.17339739e-03  3.20735127e-02  1.63787436e-02
 -1.21909454e-02  3.36886533e-02 -3.95951839e-03 -2.92740110e-02
 -3.17751095e-02 -2.94292010e-02 -1.89089794e-02 -1.53617440e-02
  5.47915958e-02  1.77100431e-02  7.65269436e-03 -3.60980891e-02
 -2.89490633e-03  1.78972594e-02  5.59852086e-03  7.96505716e-03
 -4.31313664e-02  1.45536

In [26]:
from minsearch import VectorSearch

dez_vindex = VectorSearch()
dez_vindex.fit(dez_embeddings, de_dtc_data_engineering_zoomcamp)


In [27]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
new_results = dez_vindex.search(q)


In [28]:
evidently_embeddings = []

for d in tqdm(evidently_chunks):
    print(d)
    v = embedding_model.encode(d['section'])
    evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)


  0%|          | 0/787 [00:00<?, ?it/s]

{'title': 'Create Plant', 'openapi': 'POST /plants', 'filename': 'docs-main/api-reference/endpoint/create.mdx', 'section': "It appears that you've referenced a document but haven't provided any content for it. If you share the text or details from the document, I can assist you in organizing it into logical sections for a Q&A system. Please provide the content you'd like to work with!"}
{'title': 'Delete Plant', 'openapi': 'DELETE /plants/{id}', 'filename': 'docs-main/api-reference/endpoint/delete.mdx', 'section': "Sure! Please provide the document that you'd like me to split into logical sections for a Q&A system."}
{'title': 'Get Plants', 'openapi': 'GET /plants', 'filename': 'docs-main/api-reference/endpoint/get.mdx', 'section': "## Introduction\n\nThis section provides an overview of the document's purpose and the main topics covered."}
{'title': 'Get Plants', 'openapi': 'GET /plants', 'filename': 'docs-main/api-reference/endpoint/get.mdx', 'section': '## Topic One: Key Concepts\n\

In [29]:
query = 'Can I join the course now?'

text_results = data_engineering_zoomcamp_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = dez_vindex.search(q, num_results=5)

final_results = text_results + vector_results

In [30]:
def text_search(query):
    return data_engineering_zoomcamp_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return dez_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
            
    return combined_results


In [31]:
print(hybrid_search("Am I eligible for this course ?")[0]["content"])


## Thank you!

Thanks for signing up for the course.

The process of adding you to the mailing list is not automated yet, 
but you will hear from us closer to the course start. 

To make sure you don't miss any announcements

- Register in [DataTalks.Club's Slack](https://datatalks.club/slack.html) and
  join the [`#course-data-engineering`](https://app.slack.com/client/T01ATQK62F8/C01FABYF2RG) channel
- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp)
- Subscribe to [DataTalks.Club's YouTube channel](https://www.youtube.com/c/DataTalksClub) and check 
  [the course playlist](https://www.youtube.com/playlist?list=PL3MmuxUbc_hJed7dXYoJw8DoCuVHhGEQb)

See you in January!


In [32]:
import openai

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)

It depends on the specific course you’re interested in. Many courses have flexible start dates and allow late enrollments, while others may have specific enrollment periods. 

It's best to check the course details on the official website or contact the course administrators directly for the most accurate information. If you have a specific course in mind, let me know, and I might be able to help you find the information you need!


In [33]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [34]:

system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)
print(response.output_text)


In [35]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}


In [36]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)


Yes, you can join the course now! You may want to check for any deadlines or initial requirements, as joining midway through a course might mean you need to catch up on missed materials. You can find more information and updates by registering on [DataTalks.Club's Slack](https://datatalks.club/slack.html) or joining the [course Telegram channel](https://t.me/dezoomcamp). 

If you have specific questions about the course or the content covered, feel free to ask!


In [38]:
chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)
print(response.output_text)

In [39]:
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

In [40]:
system_prompt = """
You are a helpful assistant for a course. 

Always search for relevant information before answering. 
If the first search doesn't give you enough information, try different search terms.

Make multiple searches if needed to provide comprehensive answers.
"""

In [42]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return data_engineering_zoomcamp_index.search(query, num_results=5)

In [43]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)


In [44]:
question = "I just discovered the course, can I join now?"

result = await agent.run(user_prompt=question)


In [45]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 4, 6, 18, 49, 40, 134322, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 6, 18, 49, 40, 135313, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nAlways search for relevant information before answering. \nIf the first search doesn't give you enough information, try different search terms.\n\nMake multiple searches if needed to provide comprehensive answers.", run_id='019d6421-09d6-7530-ae8f-f40dac16fcdb'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"Can I join the course now?"}', tool_call_id='call_9UV8vzmfyL2STzz2frj8ADV6')], usage=RequestUsage(input_tokens=146, output_tokens=20, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026,

In [46]:
question = "how do I install Kafka in Python?"
result = await agent.run(user_prompt=question)


In [47]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='how do I install Kafka in Python?', timestamp=datetime.datetime(2026, 4, 6, 18, 52, 53, 989307, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 6, 18, 52, 53, 989801, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nAlways search for relevant information before answering. \nIf the first search doesn't give you enough information, try different search terms.\n\nMake multiple searches if needed to provide comprehensive answers.", run_id='019d6423-ff55-70f3-b33c-85319cb37b73'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"install Kafka in Python"}', tool_call_id='call_0N5WpeInhn4M8JjKmV4x1vAX')], usage=RequestUsage(input_tokens=143, output_tokens=17, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-mini-2024-07-18', timestamp=datetime.datetime(2026, 4, 6, 18, 52, 